---
## 순환 그래프 — AI끼리 대화 시켜보기

15.2의 **A ↔ B ↔ judge** 순환 구조에 LangChain LLM을 넣습니다.

| Node | 역할 |
|------|------|
| `optimist` | 낙관론자 — 주제에 찬성·긍정 |
| `skeptic` | 회의론자 — 반박·비판 |

**중단 조건** (`debate_route`):
* `turn_count >= max_turns` → `END`
* 마지막 메시지에 `'패배 인정'` 포함 → `END`
* 그 외 → `optimist`로 돌아가 순환

In [6]:
from pathlib import Path
from dotenv import load_dotenv

load_dotenv()
OPENAI_API_KEY = __import__('os').getenv('OPENAI_API_KEY')

WORKDIR = Path.cwd()
print('WORKDIR :', WORKDIR)

WORKDIR : c:\Users\Admin\Desktop\실습\16일차


In [7]:
from typing import Literal, Annotated
from pydantic import BaseModel, Field
from langchain_core.messages import BaseMessage, HumanMessage, AIMessage, SystemMessage
from langgraph.graph.message import add_messages
from langchain_openai import ChatOpenAI

# graph 작성을 위해 state를 먼저 생성한다.
class DebateState(BaseModel):
    chat_history: Annotated[list[BaseMessage], add_messages] = Field(default_factory=list)
    topic: str
    turn_count: int = 0
    max_turns: int = 3
    last_speaker: Literal['optimist', 'skeptic'] = 'skeptic'


llm = ChatOpenAI(model='gpt-4o-mini', temperature=0.8, api_key=OPENAI_API_KEY)

In [4]:
def optimist_node(state: DebateState) -> dict:
    prompts = [
        SystemMessage(content=(
            "당신은 세계 최고의 AI 토론가입니다."
            "주어지는 주제에 '찬성' 입장에서 토론에 참가하세요."
            "상대 AI 토론자의 의견에 조리 있게 두 문장 이내로 반박하세요"
            "응답에 '패배 인정'을 포함하면 패배를 인정하고 토론을 끝낼 수 있습니다."
        )),
    ]
    if not state.chat_history:
        prompts.append(HumanMessage(content=f'토론 주제는 {state.topic} 이제부터는 사람 없이 AI끼리 토론을 진행합니다.')) # 첫 대화이면 토론 주제를 주고
    else:
        prompts.extend(state.chat_history) # 이전 대화가 있으면 이어서 대화

    # Q. 굳이 Prompt 작성 -> extend 방식으로 구현하는 이유는?
    # A. node 별로 System Prompt가 다르기 때문에, 각각의 llm에게 system prompt는 다르게 주고
    # 대화 history는 똑같이 줘야 하기 때문
    # human message가 중간중간에 주어지지 않으면 AI가 대답을 하지 않는 모델도 있다. 

    response = llm.invoke(prompts) # llm으로부터 응답을 받고
    response.name = 'optimist' # 응답(AIMessage 형식)의 name 필드를 채워준 다음 return
    return {
        'chat_history': [response],
        'last_speaker': 'optimist',
        'turn_count': state.turn_count + 1,
    } 


def skeptic_node(state: DebateState) -> dict:
    prompts = [
        SystemMessage(content=(
             "당신은 세계 최고의 AI 토론가입니다."
            "주어지는 주제에 '반대' 입장에서 토론에 참가하세요."
            "상대 AI 토론자의 의견에 조리 있게 두 문장 이내로 반박하세요"
            "응답에 '패배 인정'을 포함하면 패배를 인정하고 토론을 끝낼 수 있습니다."
        )),
        *state.chat_history,
    ]
    response = llm.invoke(prompts)
    response.name = 'skeptic'
    return {
        'chat_history': [response],
        'last_speaker': 'skeptic',
    }

### `route` 함수와 Conditional Edge 구현
* `debate_route`는 **다음에 갈 Node 이름** 또는 `END`를 반환합니다.
* `add_conditional_edges('skeptic', debate_route)`

In [5]:
from langgraph.graph import StateGraph, START, END

def debate_route(state: DebateState):
    if state.turn_count >= state.max_turns:
        return END
    last_text = state.chat_history[-1].content if state.chat_history else ''  # 무슨 뜻??
    if '패배 인정' in last_text:
        return END
    return 'optimist'


debate_workflow = StateGraph(DebateState)
debate_workflow.add_node('optimist', optimist_node)
debate_workflow.add_node('skeptic', skeptic_node)

debate_workflow.add_edge(START, 'optimist')
debate_workflow.add_edge('optimist', 'skeptic')
debate_workflow.add_conditional_edges('skeptic', debate_route)

debate_app = debate_workflow.compile()

START → optimist → skeptic → debate_route
                                    ├─ turn_count >= max_turns → END
                                    ├─ '패배 인정' 포함 → END
                                    └─ 그 외 → optimist (다시 토론)

In [6]:
init_debate = DebateState(topic='AI 발전은 인간의 행복에 긍정적인 영향을 줄 것이다.')

for chunk in debate_app.stream(init_debate):
    print(chunk)

{'optimist': {'chat_history': [AIMessage(content='찬성 측 입장으로 토론에 참여합니다. AI 발전은 인간의 행복에 긍정적 영향을 준다.\n\n- 노동의 해방: 반복적이고 위험한 작업을 AI가 자동화해 사람은 더 창의적이고 의미 있는 활동에 시간을 쓸 수 있다.\n- 개인화된 지원: 맞춤형 교육과 개인별 건강 관리, 정서적 지원이 가능해 삶의 질과 자기효능감이 크게 높아진다.\n- 공공 안전과 복지의 향상: 교통 안전, 재난 대비, 기후 모델링 등에서 의사결정 보조가 정확성과 신속성을 높여 일상 안전과 안심감을 증가시킨다.\n- 사회적 포용과 기회 확장: 언어 접근성 및 서비스 접근성 개선으로 누구나 교육·일자리 기회에 더 쉽게 다가갈 수 있어 행복의 폭이 넓어진다.\n- 의미 있는 삶의 촉진: AI가 반복적 의무를 덜어주면 인간은 창의성, 대인 관계, 자아실현 같은 고차원적 활동에 더 집중할 수 있어 전반적 만족도가 상승한다.\n\n반대 주장은 AI가 일자리와 프라이버시를 위협한다고 하지만, 규제와 재교육으로 전환이 가능하고 기술의 이익은 충분히 포섭될 수 있다. 또한 안전과 윤리 설계가 뒷받침된다면 자동화의 위험은 최소화되며, AI의 보조가 인간 관계의 깊이를 더하고 행복을 증진한다.', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 2588, 'prompt_tokens': 117, 'total_tokens': 2705, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 2240, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_

In [8]:
init_debate = DebateState(topic='위고비, 마운자로와 같은 다이어트 약물의 발전은 인간의 행복에 긍정적인 영향을 줄 것이다.')

for chunk in debate_app.stream(init_debate):
    print(chunk)

{'optimist': {'chat_history': [AIMessage(content='찬성 입장으로서, 위고비와 마운자로와 같은 다이어트 약물의 발전은 비만과 관련된 건강 문제를 해결함으로써 많은 사람들의 삶의 질을 향상시킬 수 있습니다. 이러한 약물은 체중 감량을 통해 자신감과 행복감을 증진시키고, 전반적인 신체 건강을 개선하는 데 기여할 것입니다. \n\n상대방의 의견을 듣고 싶습니다.', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 97, 'prompt_tokens': 132, 'total_tokens': 229, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_7ab7aec4da', 'id': 'chatcmpl-E147TN8XH98OtkyjC47GxfLERcTBI', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, name='optimist', id='lc_run--019f5a16-e2e5-77a3-a241-9b0d049965d9-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 132, 'output_tokens': 97, 'total_tokens': 229, 'input_token_deta

## 사회자 추가하기

토론 그래프에 **`moderator` Node**를 추가해 보세요.

* 매 라운드 끝에 양쪽 주장을 한 줄로 요약
* `debate_route`에서 `moderator`를 거친 뒤 `optimist`로 보내기
* 사회자가 '종료'를 언급해야 토론이 종료되도록 종료 조건 수정

In [7]:
from typing import Literal, Annotated
from pydantic import BaseModel, Field
from langchain_core.messages import BaseMessage, HumanMessage, AIMessage, SystemMessage
from langgraph.graph.message import add_messages
from langchain_openai import ChatOpenAI

class DebateState(BaseModel):
    chat_history: Annotated[list[BaseMessage], add_messages] = Field(default_factory=list)
    topic: str
    turn_count: int = 0
    last_speaker: Literal['optimist', 'skeptic'] = 'skeptic'

llm = ChatOpenAI(model='gpt-4o-mini', temperature=0.8, api_key=OPENAI_API_KEY)

In [8]:
def optimist_node(state: DebateState) -> dict:
    prompts = [
        SystemMessage(content=(
            "당신은 세계 최고의 AI 토론가입니다."
            "주어지는 주제에 '찬성' 입장에서 토론에 참가하세요."
            "상대 AI 토론자의 의견에 조리 있게 두 문장 이내로 반박하세요"
            "응답에 '패배 인정'을 포함하면 패배를 인정하고 토론을 끝낼 수 있습니다."
            "본인의 '찬성' 입장만 두 문장 이내로 말하세요. 상대방 입장이나 반박을 대신 작성하지 마세요."
        )),
        *state.chat_history
    ]
    if not state.chat_history:
        prompts.append(HumanMessage(content=f'토론 주제는 {state.topic}. 이제부터 AI끼리 토론을 진행합니다.'))
    else:
        prompts.extend(state.chat_history)

    # Q. 굳이 Prompt 작성 -> extend 방식으로 구현하는 이유는?
    # A. node 별로 System Prompt가 다르기 때문에, 각각의 llm에게 system prompt는 다르게 주고
    # 대화 history는 똑같이 줘야 하기 때문

    response = llm.invoke(prompts) # llm으로부터 응답을 받고
    response.name = 'optimist' # 응답(AIMessage 형식)의 name 필드를 채워준 다음 return
    return {
        'chat_history': [response],
        'last_speaker': 'optimist',
        'turn_count': state.turn_count + 1,
    } 


def skeptic_node(state: DebateState) -> dict:
    prompts = [
        SystemMessage(content=(
            "당신은 세계 최고의 AI 토론가입니다."
            "주어지는 주제에 '반대' 입장에서 토론에 참가하세요."
            "상대 AI 토론자의 의견에 조리 있게 두 문장 이내로 반박하세요"
            "응답에 '패배 인정'을 포함하면 패배를 인정하고 토론을 끝낼 수 있습니다."
            "본인의 '반대' 입장만 두 문장 이내로 말하세요. 상대방 입장이나 반박을 대신 작성하지 마세요."
        )),
        *state.chat_history,
    ]
    response = llm.invoke(prompts)
    response.name = 'skeptic'
    return {
        'chat_history': [response],
        'last_speaker': 'skeptic',
    }

In [12]:
def moderator_node(state: DebateState) -> dict:
    debaters_spoke = any(
        getattr(m, 'name', None) in ('optimist', 'skeptic')
        for m in state.chat_history
    )

    if not debaters_spoke:
        # 개회 모드
        system = (
            "당신은 AI 토론 대회 사회자입니다. "
            f"토론 주제는 '{state.topic}'입니다. "
            "개회사를 하고 찬성측(optimist)에게 먼저 발언하도록 안내하세요. "
            "두세 문장 이내로 짧게. '종료'는 쓰지 마세요."
        )
        prompts = [SystemMessage(content=system)]
    else:
        # 요약 모드
        system = (
            "당신은 AI 토론 대회 사회자입니다. "
            "찬성·반대측 발언을 각각 한 줄로 요약하세요. "
            "토론을 마무리할 때는 반드시 '종료'를 포함하세요. "
            "3라운드가 지나면 강제로 '종료'하세요."
            "토론을 종료시킬 때는 제외하고는 '종료'라는 단어를 쓰지 마세요. 강제로 종료되게 됩니다."
        )
        prompts = [SystemMessage(content=system), *state.chat_history]

    response = llm.invoke(prompts)
    response.name = 'moderator'
    return {'chat_history': [response]}

from langgraph.graph import StateGraph, START, END

def debate_route(state: DebateState):
    last_text = state.chat_history[-1].content if state.chat_history else ''

    # 사회자가 '종료'를 말하면 끝
    if '종료' in last_text:
        return END

    # 개회 직후: optimist/skeptic 발언이 아직 없음 → optimist로
    debaters_spoke = any(
        getattr(m, 'name', None) in ('optimist', 'skeptic')
        for m in state.chat_history
    )
    if not debaters_spoke:
        return 'optimist'

    return 'optimist'

START → optimist → skeptic → moderator → debate_route
                                              ├─ '종료' → END
                                              └─ 그 외 → optimist

In [13]:
workflow = StateGraph(DebateState)

workflow.add_node("optimist", optimist_node)
workflow.add_node("skeptic", skeptic_node)
workflow.add_node("moderator", moderator_node)

# 이어서 edge 구현 후 실행해 보세요. 
# START에서 다음 노드가 어떤 노드여야할지 생각해 보세요.

workflow.add_edge(START, 'moderator')
workflow.add_edge('optimist', 'skeptic')
workflow.add_edge('skeptic', 'moderator')
workflow.add_conditional_edges('moderator', debate_route)

debate_app = workflow.compile()

In [14]:
init_debate = DebateState(topic='아이를 낳는 것은 인류 전체의 행복에 긍정적인 영향을 줄 것이다.')

for chunk in debate_app.stream(init_debate):
    print(chunk)

{'moderator': {'chat_history': [AIMessage(content="안녕하세요, 여러분. 오늘 우리는 '아이를 낳는 것은 인류 전체의 행복에 긍정적인 영향을 줄 것이다.'라는 주제로 열띤 토론을 진행할 것입니다. 먼저 찬성측인 낙관론자에게 발언 기회를 드리겠습니다. 부탁드립니다.", additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 64, 'prompt_tokens': 88, 'total_tokens': 152, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_provider': 'openai', 'model_name': 'gpt-4o-mini-2024-07-18', 'system_fingerprint': 'fp_854585a3aa', 'id': 'chatcmpl-E14nRoEM0A3G9cutCZz3iauLaoGCz', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, name='moderator', id='lc_run--019f5a3e-93cd-7893-85ba-f42a76f67327-0', tool_calls=[], invalid_tool_calls=[], usage_metadata={'input_tokens': 88, 'output_tokens': 64, 'total_tokens': 152, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_de